# MMCAD Sketch Encoder: BRep-Anchored ViT Training

Trains a **ViT-base** sketch encoder to match the BRep embedding space from the v4 tri-modal model.

**Phase 1** (run once): Pre-compute BRep + text embeddings from v4 checkpoint, save to disk.
**Phase 2** (main training): Train ViT sketch encoder against frozen BRep anchors.

| Component | Details |
|---|---|
| Architecture | `google/vit-base-patch16-224` CLS token + 2-layer projection head -> 768d |
| Loss | Matryoshka anchor alignment (cosine + InfoNCE) at [128, 256, 512, 768]d |
| Temperature | 0.05 fixed (matching v4 final tau) |
| Data | Sketch ISO views (sketch_iso1 / sketch_iso2) from ABC dataset |


In [ ]:
# Cell 1: Environment Setup
import sys
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    get_ipython().system('pip install -q plyfile h5py transformers sentence-transformers pillow tqdm psutil')
    print('Colab environment ready')
else:
    print('Local environment')


In [ ]:
# Cell 2: Extract sketch images for training
import os, zipfile, time

if IS_COLAB:
    DRIVE_DIR = '/content/drive/MyDrive/MMCAD'
    EXTRACT_DIR = '/content/mmcad_data'
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    # Resume-aware: check if already extracted
    SKETCH_ROOT = None
    for candidate in [EXTRACT_DIR, DRIVE_DIR]:
        sketch_dir = os.path.join(candidate, 'sketches')
        if os.path.isdir(sketch_dir) and len(os.listdir(sketch_dir)) > 100:
            SKETCH_ROOT = candidate
            n = len(os.listdir(sketch_dir))
            print(f'Sketches already available at: {sketch_dir} ({n} files)')
            break

    if SKETCH_ROOT is None:
        for archive_name in ['sketches.zip', 'sketch_images.zip']:
            archive = f'{DRIVE_DIR}/{archive_name}'
            if os.path.exists(archive):
                print(f'Extracting {archive_name}...')
                t0 = time.time()
                with zipfile.ZipFile(archive, 'r') as z:
                    z.extractall(EXTRACT_DIR)
                print(f'Done in {time.time()-t0:.0f}s')
                SKETCH_ROOT = EXTRACT_DIR
                break

    if SKETCH_ROOT is None:
        # Try Drive directly
        if os.path.isdir(f'{DRIVE_DIR}/sketches'):
            SKETCH_ROOT = DRIVE_DIR
        else:
            print('WARNING: No sketch images found!')
            print('Expected: sketches.zip on Drive, or sketches/ directory')
            SKETCH_ROOT = EXTRACT_DIR  # will fail gracefully in dataset

    print(f'SKETCH_ROOT: {SKETCH_ROOT}')
else:
    SKETCH_ROOT = r'C:\Users\anush\Desktop\MMCAD'
    print(f'SKETCH_ROOT: {SKETCH_ROOT}')


In [ ]:
# Cell 3: Imports
import os, gc, time, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
import h5py

print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')


In [ ]:
# Cell 4: Configuration
class Config:
    # === PATHS ===
    if IS_COLAB:
        DATA_ROOT = '/content/mmcad_data'
        DRIVE_DIR = '/content/drive/MyDrive/MMCAD'
        CSV_PATH = f'{DRIVE_DIR}/abc_dataset_clean.csv'
        _local_h5 = f'{DATA_ROOT}/brep_features.h5'
        _drive_h5 = f'{DRIVE_DIR}/brep_features.h5'
        BREP_H5 = _local_h5 if os.path.exists(_local_h5) else _drive_h5
        _local_topo = f'{DATA_ROOT}/brep_topology.h5'
        _drive_topo = f'{DRIVE_DIR}/brep_topology.h5'
        BREP_TOPO_H5 = _local_topo if os.path.exists(_local_topo) else _drive_topo
        PLY_H5 = f'{DRIVE_DIR}/abc_point_clouds.h5'
        V4_CKPT = f'{DRIVE_DIR}/baseline_trimodal_v4/latest.pth'
        ANCHOR_DIR = f'{DRIVE_DIR}/sketch_anchors'
        SAVE_DIR = f'{DRIVE_DIR}/sketch_encoder_v1'
        LOCAL_SAVE_DIR = '/content/sketch_v1_ckpts'
    else:
        DATA_ROOT = r'C:\Users\anush\Desktop\MMCAD'
        CSV_PATH = os.path.join(DATA_ROOT, 'abc_dataset_clean.csv')
        BREP_H5 = os.path.join(DATA_ROOT, 'brep_features.h5')
        BREP_TOPO_H5 = os.path.join(DATA_ROOT, 'brep_topology.h5')
        PLY_H5 = os.path.join(DATA_ROOT, 'abc_point_clouds.h5')
        V4_CKPT = os.path.join(DATA_ROOT, 'baseline_trimodal_v4', 'latest.pth')
        ANCHOR_DIR = os.path.join(DATA_ROOT, 'sketch_anchors')
        SAVE_DIR = os.path.join(DATA_ROOT, 'sketch_encoder_v1')
        LOCAL_SAVE_DIR = SAVE_DIR

    # === SHARED SPACE (matches v4 exactly) ===
    D_SHARED = 768
    MATRYOSHKA_DIMS = [128, 256, 512, 768]
    MATRYOSHKA_WEIGHTS = [1.0, 1.0, 1.0, 1.5]

    # === V4 MODEL CONFIG (needed for Phase 1 anchor extraction) ===
    FACE_DIM = 16
    D_BREP_MODEL = 512
    N_BREP_LAYERS = 6
    N_HEADS = 8
    MAX_FACES = 192
    MAX_EDGES = 512
    NUM_POINTS = 2048
    DGCNN_K = 20
    DGCNN_LATENT = 1024
    TEXT_ENCODER_TYPE = 'embeddinggemma'
    TEXT_MODEL_GEMMA = 'google/embeddinggemma-300m'
    TEXT_MODEL_BERT = 'bert-base-uncased'
    TEXT_MAX_LENGTH = 512

    # === SKETCH ENCODER CONFIG ===
    VIT_MODEL = 'google/vit-base-patch16-224'
    IMAGE_SIZE = 224

    # === TRAINING ===
    BATCH_SIZE = 256
    VIT_LR = 1e-5       # ViT backbone (pretrained, lower LR)
    PROJ_LR = 1e-4      # projection head (new, higher LR)
    WEIGHT_DECAY = 0.01
    NUM_EPOCHS = 15
    WARMUP_EPOCHS = 2
    TEMPERATURE = 0.05   # FIXED: matches v4 final tau
    GRAD_CLIP = 1.0

    # === MISC ===
    NUM_WORKERS = 2  # 2 is optimal on Colab
    VAL_SPLIT = 0.1
    SEED = 42
    K_VALUES = [1, 5, 10]
    SAVE_EVERY = 5

config = Config()
os.makedirs(config.ANCHOR_DIR, exist_ok=True)
os.makedirs(config.SAVE_DIR, exist_ok=True)
os.makedirs(config.LOCAL_SAVE_DIR, exist_ok=True)
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {device}')
print(f'Anchor dir: {config.ANCHOR_DIR}')
print(f'Save dir:   {config.SAVE_DIR}')
print(f'ViT: {config.VIT_MODEL}')
print(f'Temperature: {config.TEMPERATURE} (fixed)')
print(f'LR: backbone={config.VIT_LR:.0e}, proj={config.PROJ_LR:.0e}')
print(f'Epochs: {config.NUM_EPOCHS}, batch: {config.BATCH_SIZE}')



---
## Phase 1: Extract BRep + Text Anchors (Run Once)

Loads the v4 tri-modal checkpoint, computes raw (unnormalized) BRep and text embeddings
for all valid samples, and saves them to disk. After this, the full model is deleted
from memory. **Skip these cells if anchors already exist.**


In [ ]:
# Cell 6: V4 Model Definitions (needed to load checkpoint)
# These are identical to mmcad_baseline_trimodal_v4.ipynb cells 5-8.
# Only used in Phase 1 for anchor extraction; not needed in Phase 2.

# --- DGCNN ---
def knn(x, k):
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    return (-xx - inner - xx.transpose(2, 1)).topk(k=k, dim=-1)[1]

def get_graph_feature(x, k=20):
    bs, d, n = x.size()
    idx = knn(x, k)
    idx_base = torch.arange(0, bs, device=x.device).view(-1, 1, 1) * n
    idx = (idx + idx_base).view(-1)
    x = x.transpose(2, 1).contiguous()
    feat = x.view(bs * n, -1)[idx].view(bs, n, k, d)
    x = x.view(bs, n, 1, d).repeat(1, 1, k, 1)
    return torch.cat((feat - x, x), dim=3).permute(0, 3, 1, 2).contiguous()

class DGCNNEncoder(nn.Module):
    def __init__(self, latent_size=1024, k=20):
        super().__init__()
        self.k = k
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d(256)
        self.bn5 = nn.BatchNorm1d(latent_size)
        self.bn6 = nn.BatchNorm1d(512)
        self.bn7 = nn.BatchNorm1d(latent_size)
        self.conv1 = nn.Sequential(nn.Conv2d(6, 64, 1, bias=False), self.bn1, nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(128, 64, 1, bias=False), self.bn2, nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(128, 128, 1, bias=False), self.bn3, nn.LeakyReLU(0.2))
        self.conv4 = nn.Sequential(nn.Conv2d(256, 256, 1, bias=False), self.bn4, nn.LeakyReLU(0.2))
        self.conv5 = nn.Sequential(nn.Conv1d(512, latent_size, 1, bias=False), self.bn5, nn.LeakyReLU(0.2))
        self.linear1 = nn.Linear(latent_size * 2, 512, bias=False)
        self.linear2 = nn.Linear(512, latent_size)
        self.dp = nn.Dropout(0.3)
    def forward(self, x):
        x1 = self.conv1(get_graph_feature(x, self.k)).max(dim=-1)[0]
        x2 = self.conv2(get_graph_feature(x1, self.k)).max(dim=-1)[0]
        x3 = self.conv3(get_graph_feature(x2, self.k)).max(dim=-1)[0]
        x4 = self.conv4(get_graph_feature(x3, self.k)).max(dim=-1)[0]
        x = self.conv5(torch.cat((x1, x2, x3, x4), dim=1))
        x = torch.cat((x.max(2)[0], x.mean(2)), 1)
        x = self.dp(F.leaky_relu(self.bn6(self.linear1(x)), 0.2))
        return self.dp(self.bn7(self.linear2(x)))

# --- BRepFormer ---
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return self.scale * x / torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)

class SwiGLUFFN(nn.Module):
    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.w1 = nn.Linear(d_in, d_hidden, bias=True)
        self.w2 = nn.Linear(d_hidden, d_in, bias=True)
        self.w3 = nn.Linear(d_in, d_hidden, bias=True)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class BRepFormerLayer(nn.Module):
    def __init__(self, d_model=256, n_heads=8, ffn_mult=4, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = SwiGLUFFN(d_model, d_model * ffn_mult)
        self.attn_drop = nn.Dropout(dropout)
    def forward(self, x, attn_bias=None, mask=None):
        B, N, D = x.shape
        h = self.norm1(x)
        q = self.q_proj(h).reshape(B, N, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        k = self.k_proj(h).reshape(B, N, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        v = self.v_proj(h).reshape(B, N, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if attn_bias is not None: attn = attn + attn_bias
        if mask is not None: attn = attn.masked_fill(mask[:, None, None, :] == 0, float('-inf'))
        attn = self.attn_drop(F.softmax(attn, dim=-1))
        out = (attn @ v).transpose(1, 2).reshape(B, N, D)
        x = x + self.out_proj(out)
        x = x + self.ffn(self.norm2(x))
        return x

def compute_topo_distances(edge_to_faces, face_centroids, face_mask, face_normals, max_faces):
    B = edge_to_faces.shape[0]
    N_f = max_faces
    device = edge_to_faces.device
    distances = torch.zeros(B, N_f, N_f, 3, device=device)
    for b in range(B):
        n_f = max(int(face_mask[b].sum().item()), 1)
        e2f = edge_to_faces[b].long()
        adj = [[] for _ in range(n_f)]
        for i in range(e2f.shape[0]):
            f0, f1 = e2f[i, 0].item(), e2f[i, 1].item()
            if 0 <= f0 < n_f and 0 <= f1 < n_f and f0 != f1:
                adj[f0].append(f1); adj[f1].append(f0)
        sp = torch.full((n_f, n_f), float('inf'), device=device)
        for src in range(n_f):
            sp[src, src] = 0
            queue, head = [src], 0
            while head < len(queue):
                u = queue[head]; head += 1
                for v in adj[u]:
                    if sp[src, v] == float('inf'):
                        sp[src, v] = sp[src, u] + 1; queue.append(v)
        finite = sp != float('inf')
        if finite.any():
            mx = sp[finite].max(); sp[~finite] = mx + 1; sp = sp / (mx + 1)
        distances[b, :n_f, :n_f, 0] = sp
        centroids = face_centroids[b, :n_f]
        cdist = torch.cdist(centroids.unsqueeze(0), centroids.unsqueeze(0)).squeeze(0)
        diag = (centroids.max(0).values - centroids.min(0).values).norm() + 1e-8
        distances[b, :n_f, :n_f, 1] = cdist / diag
        normals = face_normals[b, :n_f]
        n_norms = normals.norm(dim=1, keepdim=True).clamp(min=1e-8)
        normals_unit = normals / n_norms
        cos_sim = torch.mm(normals_unit, normals_unit.T).clamp(-1.0, 1.0)
        distances[b, :n_f, :n_f, 2] = torch.acos(cos_sim) / math.pi
    return distances

class BRepFormerEncoder(nn.Module):
    def __init__(self, d_face_in=16, d_out=768, d_model=256, n_heads=8, n_layers=6, max_faces=192):
        super().__init__()
        self.d_model = d_model
        self.max_faces = max_faces
        self.face_proj = nn.Linear(d_face_in, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.topo_bias_proj = nn.Sequential(
            nn.Linear(3, d_model), RMSNorm(d_model), nn.ReLU(), nn.Linear(d_model, n_heads))
        self.layers = nn.ModuleList([BRepFormerLayer(d_model, n_heads) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.output_proj = nn.Sequential(nn.Linear(d_model, d_out), nn.GELU(), nn.Linear(d_out, d_out))
    def forward(self, face_features, face_centroids, edge_to_faces, face_mask, face_normals):
        B, N_f = face_features.shape[:2]
        x = self.face_proj(face_features)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        topo_dist = compute_topo_distances(edge_to_faces, face_centroids, face_mask, face_normals, self.max_faces)
        topo_padded = F.pad(topo_dist, (0, 0, 1, 0, 1, 0), value=0)
        attn_bias = self.topo_bias_proj(topo_padded).permute(0, 3, 1, 2)
        mask = torch.cat([torch.ones(B, 1, device=face_features.device), face_mask], dim=1)
        for layer in self.layers:
            x = layer(x, attn_bias=attn_bias, mask=mask)
        return self.output_proj(self.final_norm(x)[:, 0, :])

# --- Text Encoders ---
class EmbeddingGemmaEncoder(nn.Module):
    def __init__(self, model_id='google/embeddinggemma-300m'):
        super().__init__()
        from sentence_transformers import SentenceTransformer
        print(f'Loading EmbeddingGemma: {model_id}')
        st_model = SentenceTransformer(model_id, trust_remote_code=True)
        self._pipeline = nn.ModuleList(list(st_model))
        self.tokenizer = st_model.tokenizer
        for p in self.parameters(): p.requires_grad = True
        n_params = sum(p.numel() for p in self.parameters()) / 1e6
        print(f'  Pipeline: {len(self._pipeline)} modules, {n_params:.1f}M params')
    def forward(self, input_ids, attention_mask):
        features = {'input_ids': input_ids, 'attention_mask': attention_mask}
        for module in self._pipeline:
            features = module(features)
        return features['sentence_embedding']

class BertTextEncoder(nn.Module):
    def __init__(self, model_name='bert-base-uncased', d_out=768):
        super().__init__()
        from transformers import BertModel, BertTokenizer
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.bert = BertModel.from_pretrained(model_name)
        self.projection = nn.Linear(768, d_out)
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.projection(out.last_hidden_state[:, 0, :])

# --- DGCNNWithProjection ---
class DGCNNWithProjection(nn.Module):
    def __init__(self, d_out=768, k=20, latent_size=1024):
        super().__init__()
        self.dgcnn = DGCNNEncoder(latent_size=latent_size, k=k)
        self.projection = nn.Sequential(
            nn.Linear(latent_size, d_out), nn.LayerNorm(d_out),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(d_out, d_out))
    def forward(self, x):
        return self.projection(self.dgcnn(x))

# --- Baseline Model ---
class Baseline(nn.Module):
    def __init__(self, text_encoder, config):
        super().__init__()
        self.text_encoder = text_encoder
        self.brep_encoder = BRepFormerEncoder(
            d_face_in=config.FACE_DIM, d_out=config.D_SHARED,
            d_model=config.D_BREP_MODEL, n_heads=config.N_HEADS,
            n_layers=config.N_BREP_LAYERS, max_faces=config.MAX_FACES)
        self.pc_encoder = DGCNNWithProjection(
            d_out=config.D_SHARED, k=config.DGCNN_K, latent_size=config.DGCNN_LATENT)
        self.log_tau = nn.Parameter(torch.log(torch.tensor(0.07)))
        self.matryoshka_dims = config.MATRYOSHKA_DIMS
    @property
    def tau(self): return self.log_tau.exp().clamp(0.05, 1.0)
    def forward(self, batch):
        dev = next(self.parameters()).device
        z_text = self.text_encoder(batch['input_ids'].to(dev), batch['attention_mask'].to(dev))
        z_brep = self.brep_encoder(
            batch['face_features'].to(dev), batch['face_centroids'].to(dev),
            batch['edge_to_faces'].to(dev), batch['face_mask'].to(dev),
            batch['face_normals'].to(dev))
        z_pc = self.pc_encoder(batch['point_cloud'].to(dev))
        return {'z_text': z_text, 'z_brep': z_brep, 'z_pc': z_pc, 'tau': self.tau}

print('Phase 1 model definitions loaded (DGCNN + BRepFormer + Text + Baseline).')


In [ ]:
# Cell 7: HuggingFace Login (needed for EmbeddingGemma)
from huggingface_hub import login
login()


In [ ]:
# Cell 8: Pre-compute BRep + Text anchors (scipy-accelerated)
# Run ONCE, saves to Drive, then skip on future runs.

import psutil
from scipy.sparse import lil_matrix
from scipy.sparse.csgraph import shortest_path as sp_shortest_path
from concurrent.futures import ThreadPoolExecutor

# === Override compute_topo_distances with C-backed scipy version ===
# Original Python BFS: ~100x slower. This override is picked up by
# BRepFormerEncoder.forward() because both share __main__ globals.

def _bfs_one_sample(args):
    'All-pairs shortest path for one sample via scipy (C implementation).'
    e2f, n_f, N_f = args
    adj = lil_matrix((n_f, n_f), dtype=np.float32)
    for i in range(e2f.shape[0]):
        f0, f1 = int(e2f[i, 0]), int(e2f[i, 1])
        if 0 <= f0 < n_f and 0 <= f1 < n_f and f0 != f1:
            adj[f0, f1] = 1; adj[f1, f0] = 1
    sp = sp_shortest_path(adj.tocsr(), method='D', unweighted=True)
    finite = sp != np.inf
    if finite.any():
        mx = sp[finite].max()
        sp[~finite] = mx + 1
        if mx > 0: sp /= (mx + 1)
    else:
        sp[:] = 0
    out = np.zeros((N_f, N_f), dtype=np.float32)
    out[:n_f, :n_f] = sp
    return out


def compute_topo_distances(edge_to_faces, face_centroids, face_mask, face_normals, max_faces):
    'Scipy-accelerated topo distances. BFS via C + threads, Ch1/Ch2 via numpy.'
    B = edge_to_faces.shape[0]
    N_f = max_faces
    device = edge_to_faces.device
    e2f_np = edge_to_faces.cpu().numpy()
    fc_np = face_centroids.cpu().numpy()
    fm_np = face_mask.cpu().numpy()
    fn_np = face_normals.cpu().numpy()
    n_fs = [max(int(fm_np[b].sum()), 1) for b in range(B)]

    # Ch0: Parallel BFS (scipy releases GIL -> real thread parallelism)
    bfs_args = [(e2f_np[b], n_fs[b], N_f) for b in range(B)]
    with ThreadPoolExecutor(max_workers=min(8, B)) as pool:
        bfs_results = list(pool.map(_bfs_one_sample, bfs_args))

    out = np.zeros((B, N_f, N_f, 3), dtype=np.float32)
    for b in range(B):
        out[b, :, :, 0] = bfs_results[b]
        n_f = n_fs[b]
        # Ch1: Centroid Euclidean distance
        c = fc_np[b, :n_f]
        cdist_mat = np.linalg.norm(c[:, None] - c[None, :], axis=-1)
        diag = np.linalg.norm(c.max(0) - c.min(0)) + 1e-8
        out[b, :n_f, :n_f, 1] = cdist_mat / diag
        # Ch2: Angular distance
        n = fn_np[b, :n_f]
        norms = np.maximum(np.linalg.norm(n, axis=1, keepdims=True), 1e-8)
        n_unit = n / norms
        cos_sim = np.clip(n_unit @ n_unit.T, -1.0, 1.0)
        out[b, :n_f, :n_f, 2] = np.arccos(cos_sim) / np.pi
    return torch.from_numpy(out).to(device)

print('compute_topo_distances overridden with scipy-accelerated version (C BFS + threads)')

# === Anchor paths ===
BREP_ANCHOR_PATH = os.path.join(config.ANCHOR_DIR, 'brep_anchors.pt')
SPLIT_PATH = os.path.join(config.ANCHOR_DIR, 'split_info.pt')

if os.path.exists(BREP_ANCHOR_PATH) and os.path.exists(SPLIT_PATH):
    print(f'Anchors already exist at {config.ANCHOR_DIR}')
    print('Skipping Phase 1. Jump to Phase 2.')
else:
    print('=== Phase 1: Extracting BRep + Text anchors ===')
    t0_phase = time.time()

    # --- Load CSV + pre-build text lookup (O(1) per UID) ---
    df = pd.read_csv(config.CSV_PATH)
    df['uid_str'] = df['uid'].astype(str).str.strip()
    print(f'CSV: {len(df)} rows')


    # --- Load BRep H5 ---
    compact_cache = {}
    with h5py.File(config.BREP_H5, 'r') as f:
        for k in ['face_features', 'face_masks', 'edge_to_faces', 'face_centroids', 'uids']:
            compact_cache[k] = f[k][:]
    brep_uid_to_idx = {}
    for i, u in enumerate(compact_cache['uids']):
        k = u.decode('utf-8') if isinstance(u, bytes) else str(u)
        brep_uid_to_idx[k.strip()] = i
    print(f'BRep: {len(brep_uid_to_idx)} samples')

    # --- Load topo (face normals) ---
    topo_cache = None; fn_uid_to_idx = {}
    if os.path.exists(config.BREP_TOPO_H5):
        topo_cache = {}
        with h5py.File(config.BREP_TOPO_H5, 'r') as f:
            if 'face_normals' in f: topo_cache['face_normals'] = f['face_normals'][:]
            topo_cache['uids'] = f['uids'][:]
        for i, u in enumerate(topo_cache['uids']):
            k = u.decode('utf-8') if isinstance(u, bytes) else str(u)
            fn_uid_to_idx[k.strip()] = i
        print('Topo: face_normals loaded')

    # --- Load PLY UIDs for split reconstruction ---
    ply_uids = set()
    if os.path.exists(config.PLY_H5):
        with h5py.File(config.PLY_H5, 'r') as f:
            uid_key = next((k for k in f.keys() if 'uid' in k.lower()), None)
            if uid_key:
                for u in f[uid_key][:]:
                    k = u.decode('utf-8') if isinstance(u, bytes) else str(u)
                    ply_uids.add(k.strip())
        print(f'PLY UIDs: {len(ply_uids)}')
    else:
        ply_uids = set(df[df['has_ply'] == True]['uid_str'].tolist())
        print(f'PLY UIDs from CSV: {len(ply_uids)}')

    # --- Reconstruct exact v4 train/val split ---
    df_valid = df[df['has_ply'] == True].copy()
    df_valid['uid_str'] = df_valid['uid'].astype(str).str.strip()
    df_valid = df_valid[df_valid['uid_str'].isin(ply_uids)].reset_index(drop=True)
    n_val = int(len(df_valid) * config.VAL_SPLIT)
    indices = np.random.RandomState(config.SEED).permutation(len(df_valid))
    df_train = df_valid.iloc[indices[n_val:]].reset_index(drop=True)
    df_val = df_valid.iloc[indices[:n_val]].reset_index(drop=True)
    df_train = df_train[df_train['uid_str'].isin(brep_uid_to_idx)].reset_index(drop=True)
    df_val = df_val[df_val['uid_str'].isin(brep_uid_to_idx)].reset_index(drop=True)
    print(f'Split: Train={len(df_train)}, Val={len(df_val)}')

    # --- Normalization stats ---
    ff = compact_cache['face_features']
    # Chunked float64 stats: avoids overflow AND the ~9GB RAM copy
    n_feat = ff.shape[-1]
    _sum = np.zeros(n_feat, dtype=np.float64)
    _sum2 = np.zeros(n_feat, dtype=np.float64)
    _count = 0
    for ci in range(0, len(ff), 5000):
        chunk = ff[ci:ci+5000].reshape(-1, n_feat)
        valid = chunk[np.any(chunk != 0, axis=1)].astype(np.float64)
        _sum += valid.sum(0); _sum2 += (valid**2).sum(0); _count += len(valid)
    ff_mean = (_sum / _count).astype(np.float32)
    ff_std = np.sqrt(np.maximum(_sum2/_count - (_sum/_count)**2, 0)).astype(np.float32)
    ff_std[ff_std < 1e-6] = 1.0
    del _sum, _sum2
    print(f'Norm stats: mean [{ff_mean.min():.2f}, {ff_mean.max():.2f}], std [{ff_std.min():.2f}, {ff_std.max():.2f}]')

    # --- Build model + load checkpoint ---
    try:
        text_encoder = EmbeddingGemmaEncoder(config.TEXT_MODEL_GEMMA)
        tokenizer = text_encoder.tokenizer
    except Exception as e:
        print(f'EmbeddingGemma failed: {e}, using BERT')
        text_encoder = BertTextEncoder(d_out=config.D_SHARED)
        tokenizer = text_encoder.tokenizer
    model = Baseline(text_encoder, config).to(device)
    ckpt = torch.load(config.V4_CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print(f'Loaded v4 from epoch {ckpt["epoch"]+1}')
    print(f'RAM: {psutil.virtual_memory().used/1e9:.1f} GB, VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

    # --- Helper: batch BRep data ---
    def get_brep_batch(uid_strs):
        batch = {k: [] for k in ['face_features','face_centroids','edge_to_faces','face_mask','face_normals']}
        for uid_str in uid_strs:
            idx = brep_uid_to_idx[uid_str]
            n_f = int(compact_cache['face_masks'][idx].sum())
            ff_i = (compact_cache['face_features'][idx] - ff_mean) / ff_std; ff_i[n_f:] = 0
            fc_i = compact_cache['face_centroids'][idx].copy()
            if n_f > 0:
                c = fc_i[:n_f].mean(0); s = np.abs(fc_i[:n_f] - c).max() + 1e-8
                fc_i = (fc_i - c) / s; fc_i[n_f:] = 0
            fn_i = (topo_cache['face_normals'][fn_uid_to_idx[uid_str]]
                    if topo_cache and uid_str in fn_uid_to_idx
                    else compact_cache['face_features'][idx, :, 12:15])
            batch['face_features'].append(torch.tensor(ff_i, dtype=torch.float32))
            batch['face_centroids'].append(torch.tensor(fc_i, dtype=torch.float32))
            batch['edge_to_faces'].append(torch.tensor(compact_cache['edge_to_faces'][idx], dtype=torch.long))
            batch['face_mask'].append(torch.tensor(compact_cache['face_masks'][idx], dtype=torch.float32))
            batch['face_normals'].append(torch.tensor(fn_i, dtype=torch.float32))
        return {k: torch.stack(v).to(device) for k, v in batch.items()}

    # ==============================
    # EXTRACT BREP EMBEDDINGS
    # ==============================
    all_uids = df_train['uid_str'].tolist() + df_val['uid_str'].tolist()
    brep_anchors = {}
    BREP_BS = 128  # safe for 40GB A100

    print(f'\nExtracting BRep embeddings: {len(all_uids)} samples, batch={BREP_BS}')
    t0 = time.time()
    with torch.no_grad():
        for start in tqdm(range(0, len(all_uids), BREP_BS), desc='BRep anchors'):
            batch_uids = all_uids[start:start + BREP_BS]
            brep_batch = get_brep_batch(batch_uids)
            z_brep = model.brep_encoder(
                brep_batch['face_features'], brep_batch['face_centroids'],
                brep_batch['edge_to_faces'], brep_batch['face_mask'],
                brep_batch['face_normals'])
            for i, uid in enumerate(batch_uids):
                brep_anchors[uid] = z_brep[i].cpu()
    brep_time = time.time() - t0
    print(f'BRep extraction done in {brep_time:.0f}s ({len(all_uids)/brep_time:.0f} samples/s)')

    torch.save(brep_anchors, BREP_ANCHOR_PATH)
    print(f'Saved {len(brep_anchors)} BRep anchors -> {BREP_ANCHOR_PATH}')



    # ==============================
    # SAVE SPLIT INFO
    # ==============================
    split_info = {
        'train_uids': df_train['uid_str'].tolist(),
        'val_uids': df_val['uid_str'].tolist(),
    }
    torch.save(split_info, SPLIT_PATH)
    print(f'Saved split: {len(split_info["train_uids"])} train, {len(split_info["val_uids"])} val')

    # ==============================
    # CLEANUP
    # ==============================
    del model, text_encoder, ckpt
    del compact_cache, topo_cache
    torch.cuda.empty_cache(); gc.collect()
    total_time = time.time() - t0_phase
    print(f'\nPhase 1 complete in {total_time:.0f}s')
    print(f'RAM after cleanup: {psutil.virtual_memory().used/1e9:.1f} GB')
    print(f'Files saved to {config.ANCHOR_DIR}:')
    for p in [BREP_ANCHOR_PATH, SPLIT_PATH]:
        sz = os.path.getsize(p) / 1e6
        print(f'  {os.path.basename(p)}: {sz:.0f} MB')




---
## Phase 2: Train ViT Sketch Encoder

Loads pre-computed BRep anchors and trains a ViT-base encoder to match
the BRep embedding space. Lightweight: only one small encoder in memory,
no BRep/text models needed.


In [ ]:
# Cell 10: ViT Sketch Encoder
# google/vit-base-patch16-224 CLS token + 2-layer projection head (matches DGCNNWithProjection pattern)
from transformers import ViTModel

class ViTSketchEncoder(nn.Module):
    def __init__(self, model_name='google/vit-base-patch16-224', d_out=768):
        super().__init__()
        self.vit = ViTModel.from_pretrained(model_name)
        h = self.vit.config.hidden_size  # 768 for vit-base
        self.projection = nn.Sequential(
            nn.Linear(h, d_out),
            nn.LayerNorm(d_out),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_out, d_out),
        )

    def forward(self, pixel_values):
        cls_token = self.vit(pixel_values=pixel_values).last_hidden_state[:, 0]
        return self.projection(cls_token)

n_params = sum(p.numel() for p in ViTSketchEncoder().parameters()) / 1e6
print(f'ViTSketchEncoder: {n_params:.1f}M params')


In [ ]:
# Cell 11: Sketch Dataset + Transforms

# ImageNet normalization (ViT was pretrained on ImageNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class SketchAnchorDataset(Dataset):
    'Loads sketch images + pre-computed BRep anchors. Skips corrupt images gracefully.'

    def __init__(self, df, sketch_root, brep_anchors, transform, use_both_views=True):
        self.transform = transform
        self.sketch_root = sketch_root
        self.brep_anchors = brep_anchors
        self.use_both_views = use_both_views
        self._bad_paths = set()

        df = df.copy()
        df['uid_str'] = df['uid'].astype(str).str.strip()

        # Filter: must have sketch + anchor
        valid_mask = (
            (df['has_sketch_iso1'] == True) &
            df['uid_str'].isin(set(brep_anchors.keys()))
        )
        if sketch_root:
            exists_mask = df['sketch_iso1_path'].apply(
                lambda p: os.path.exists(os.path.join(sketch_root, str(p))))
            valid_mask = valid_mask & exists_mask

        self.samples = df[valid_mask].reset_index(drop=True)
        print(f'SketchAnchorDataset: {len(self.samples)} samples '
              f'(filtered from {len(df)}, sketch_root={sketch_root})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # Retry on corrupt/unreadable images (up to 10 random replacements)
        for attempt in range(10):
            try:
                return self._load_sample(idx)
            except Exception:
                # Track and warn (suppress after 20)
                row = self.samples.iloc[idx]
                bad = str(row.get('sketch_iso1_path', f'idx_{idx}'))
                if bad not in self._bad_paths:
                    self._bad_paths.add(bad)
                    if len(self._bad_paths) <= 20:
                        print(f'WARNING: corrupt sketch #{len(self._bad_paths)}: {bad}')
                    elif len(self._bad_paths) == 21:
                        print('WARNING: suppressing further corrupt image warnings...')
                idx = np.random.randint(len(self))
        raise RuntimeError(f'Failed after 10 attempts, {len(self._bad_paths)} corrupt images')

    def _load_sample(self, idx):
        row = self.samples.iloc[idx]
        uid_str = str(row['uid']).strip()

        # Random view selection for augmentation
        use_iso2 = (self.use_both_views and
                    row.get('has_sketch_iso2', False) == True and
                    np.random.rand() > 0.5)
        img_rel = row['sketch_iso2_path'] if use_iso2 else row['sketch_iso1_path']
        img_path = os.path.join(self.sketch_root, str(img_rel))

        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            # Fallback: try the other view
            alt_rel = row['sketch_iso1_path'] if use_iso2 else row.get('sketch_iso2_path', '')
            img_path = os.path.join(self.sketch_root, str(alt_rel))
            img = Image.open(img_path).convert('RGB')

        return {
            'pixel_values': self.transform(img),
            'anchor': self.brep_anchors[uid_str],
            'uid': uid_str,
        }

print('SketchAnchorDataset defined (with corrupt image resilience).')


In [ ]:
# Cell 12: Loss Functions
# Anchor alignment: cosine pull + InfoNCE push (against in-batch negatives)
# Applied at Matryoshka scales to preserve coarse-to-fine hierarchy

def anchor_alignment_loss(z_pred, z_anchor, temperature=0.05):
    """Combined cosine + contrastive loss for aligning to anchor space.
    Pure MSE causes mode collapse (predict the mean).
    Pure InfoNCE wastes the anchor's geometric structure.
    Combined gives both metric matching and discriminative power.
    """
    z_pred_n = F.normalize(z_pred, dim=-1)
    z_anchor_n = F.normalize(z_anchor, dim=-1)

    # Cosine alignment (positive pair pull)
    cos_loss = 1.0 - (z_pred_n * z_anchor_n).sum(dim=-1).mean()

    # InfoNCE against in-batch anchors (negative pair push)
    B = z_pred.shape[0]
    logits = torch.clamp(z_pred_n @ z_anchor_n.T / temperature, -100, 100)
    labels = torch.arange(B, device=z_pred.device)
    contrastive_loss = F.cross_entropy(logits, labels)

    return cos_loss + contrastive_loss


def matryoshka_anchor_loss(z_pred, z_anchor, dims, weights, temperature=0.05):
    """Matryoshka-structured anchor alignment across multiple scales.
    Preserves the coarse-to-fine hierarchy of the BRep anchor space.
    """
    total = 0.0
    info = {}
    for dim, w in zip(dims, weights):
        loss = anchor_alignment_loss(z_pred[:, :dim], z_anchor[:, :dim], temperature)
        total += w * loss
        info[f'loss_{dim}d'] = loss.item()
    return total / sum(weights), info

print('Loss functions defined.')
print(f'Temperature: {config.TEMPERATURE} (fixed, matching v4 final tau)')


In [ ]:
# Cell 13: Training & Evaluation Functions

def train_one_epoch(model, loader, optimizer, scheduler, config):
    model.train()
    total_loss = 0.0
    n_batches = 0

    pbar = tqdm(loader, desc='Train')
    for batch in pbar:
        pixel_values = batch['pixel_values'].to(device)
        anchors = batch['anchor'].to(device)

        z_pred = model(pixel_values)
        loss, info = matryoshka_anchor_loss(
            z_pred, anchors,
            config.MATRYOSHKA_DIMS, config.MATRYOSHKA_WEIGHTS,
            config.TEMPERATURE)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP)
        optimizer.step()
        if scheduler: scheduler.step()

        total_loss += loss.item()
        n_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}',
                          f'l_{config.D_SHARED}d': f'{info.get(f"loss_{config.D_SHARED}d", 0):.4f}'})

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, loader, config):
    model.eval()
    total_loss = 0.0
    all_z, all_uids = [], []
    n_batches = 0

    for batch in tqdm(loader, desc='Eval'):
        pixel_values = batch['pixel_values'].to(device)
        anchors = batch['anchor'].to(device)

        z_pred = model(pixel_values)
        loss, _ = matryoshka_anchor_loss(
            z_pred, anchors,
            config.MATRYOSHKA_DIMS, config.MATRYOSHKA_WEIGHTS,
            config.TEMPERATURE)

        total_loss += loss.item()
        n_batches += 1
        all_z.append(z_pred.cpu())
        all_uids.extend(batch['uid'])

    return total_loss / max(n_batches, 1), torch.cat(all_z), all_uids


@torch.no_grad()
def retrieval_eval(z_query, z_gallery, k_values=[1, 5, 10]):
    """Compute R@k: for each query i, check if gallery[i] is in top-k."""
    z_q = F.normalize(z_query, dim=-1)
    z_g = F.normalize(z_gallery, dim=-1)
    sim = z_q @ z_g.T  # (N, N)
    N = sim.shape[0]

    results = {}
    for k in k_values:
        topk_idx = sim.topk(k, dim=1).indices  # (N, k)
        correct = sum(1 for i in range(N) if i in topk_idx[i])
        results[f'R@{k}'] = 100.0 * correct / N
    return results


def full_eval(sketch_model, val_loader, brep_anchors_dict, config):
    """Full evaluation: sketch->brep retrieval at all Matryoshka dims."""
    _, z_sketch_all, uids_all = evaluate(sketch_model, val_loader, config)

    # Stack anchors in same order as sketch predictions
    z_brep = torch.stack([brep_anchors_dict[u] for u in uids_all])

    results = {}
    for dim in config.MATRYOSHKA_DIMS:
        z_s = z_sketch_all[:, :dim]
        z_b = z_brep[:, :dim]

        sb = retrieval_eval(z_s, z_b, config.K_VALUES)
        for k, v in sb.items():
            results[f'sketch2brep_{k}_{dim}d'] = v

    return results

print('Training and evaluation functions defined.')



In [ ]:
# Cell 14: Load Anchors + Create Datasets & DataLoaders

# --- Load pre-computed anchors ---
BREP_ANCHOR_PATH = os.path.join(config.ANCHOR_DIR, 'brep_anchors.pt')
SPLIT_PATH = os.path.join(config.ANCHOR_DIR, 'split_info.pt')

brep_anchors = torch.load(BREP_ANCHOR_PATH, map_location='cpu', weights_only=True)
print(f'Loaded {len(brep_anchors)} BRep anchors')

split_info = torch.load(SPLIT_PATH, map_location='cpu', weights_only=True)
train_uids = set(split_info['train_uids'])
val_uids = set(split_info['val_uids'])
print(f'Split: {len(train_uids)} train, {len(val_uids)} val')

# --- Load CSV and split ---
df = pd.read_csv(config.CSV_PATH)
df['uid_str'] = df['uid'].astype(str).str.strip()

df_train = df[df['uid_str'].isin(train_uids)].reset_index(drop=True)
df_val = df[df['uid_str'].isin(val_uids)].reset_index(drop=True)

# --- Create datasets ---
train_dataset = SketchAnchorDataset(
    df_train, SKETCH_ROOT, brep_anchors, train_transform, use_both_views=True)
val_dataset = SketchAnchorDataset(
    df_val, SKETCH_ROOT, brep_anchors, val_transform, use_both_views=False)

train_loader = DataLoader(
    train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
    num_workers=config.NUM_WORKERS, pin_memory=True,
    persistent_workers=True, drop_last=True)
val_loader = DataLoader(
    val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
    num_workers=config.NUM_WORKERS, pin_memory=True,
    persistent_workers=True)

print(f'\nTrain: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} samples, {len(val_loader)} batches')





In [ ]:
# Cell 15: Create Sketch Model + Optimizer + Scheduler

sketch_model = ViTSketchEncoder(
    model_name=config.VIT_MODEL, d_out=config.D_SHARED
).to(device)

# Differential LR: lower for pretrained ViT backbone, higher for projection head
optimizer = torch.optim.AdamW([
    {'params': sketch_model.vit.parameters(), 'lr': config.VIT_LR,
     'weight_decay': config.WEIGHT_DECAY},
    {'params': sketch_model.projection.parameters(), 'lr': config.PROJ_LR,
     'weight_decay': config.WEIGHT_DECAY},
])

total_steps = len(train_loader) * config.NUM_EPOCHS
warmup_steps = len(train_loader) * config.WARMUP_EPOCHS


def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

n_params = sum(p.numel() for p in sketch_model.parameters()) / 1e6
n_trainable = sum(p.numel() for p in sketch_model.parameters() if p.requires_grad) / 1e6
print(f'ViT Sketch Encoder: {n_params:.1f}M total, {n_trainable:.1f}M trainable')
print(f'Optimizer: AdamW, backbone_lr={config.VIT_LR:.0e}, proj_lr={config.PROJ_LR:.0e}')
print(f'Scheduler: cosine with {config.WARMUP_EPOCHS} warmup epochs')
print(f'Total steps: {total_steps}, warmup: {warmup_steps}')


In [ ]:
# Cell 16: Training Loop (with resume from checkpoint)

# === Resume logic ===
start_epoch = 0
best_r1 = 0.0
history = []

# Check Drive then local for existing checkpoint
resume_ckpt = None
for ckpt_dir in [config.SAVE_DIR, config.LOCAL_SAVE_DIR]:
    for fname in ['latest.pth', 'best.pth']:
        p = os.path.join(ckpt_dir, fname)
        if os.path.exists(p):
            resume_ckpt = p
            break
    if resume_ckpt:
        break

if resume_ckpt:
    ckpt = torch.load(resume_ckpt, map_location=device, weights_only=False)
    sketch_model.load_state_dict(ckpt['model_state_dict'])
    start_epoch = ckpt['epoch'] + 1

    # Restore best R@1 from checkpoint metrics
    if 'metrics' in ckpt:
        best_r1 = ckpt['metrics'].get(f'sketch2brep_R@1_{config.D_SHARED}d', 0)

    # Try loading optimizer state from best.pth
    best_ckpt_path = os.path.join(os.path.dirname(resume_ckpt), 'best.pth')
    if os.path.exists(best_ckpt_path):
        best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
        if 'optimizer_state_dict' in best_ckpt:
            try:
                optimizer.load_state_dict(best_ckpt['optimizer_state_dict'])
                print('Restored optimizer state from best.pth')
            except Exception as e:
                print(f'Could not restore optimizer: {e}')
        if 'metrics' in best_ckpt:
            best_r1 = max(best_r1, best_ckpt['metrics'].get(f'sketch2brep_R@1_{config.D_SHARED}d', 0))
        del best_ckpt

    # Fast-forward scheduler to correct position
    steps_done = start_epoch * len(train_loader)
    for _ in range(steps_done):
        scheduler.step()

    del ckpt
    torch.cuda.empty_cache()
    print(f'Resumed from epoch {start_epoch} (checkpoint: {resume_ckpt})')
    print(f'Best sketch2brep R@1 so far: {best_r1:.2f}%')
    print(f'Remaining epochs: {start_epoch+1}-{config.NUM_EPOCHS}')
else:
    print('No checkpoint found. Starting from scratch.')

# === Training loop ===
for epoch in range(start_epoch, config.NUM_EPOCHS):
    t0 = time.time()
    print(f'\nEpoch {epoch + 1}/{config.NUM_EPOCHS}')
    print('-' * 50)

    # Train
    train_loss = train_one_epoch(sketch_model, train_loader, optimizer, scheduler, config)

    # Evaluate
    val_loss, _, _ = evaluate(sketch_model, val_loader, config)

    # Retrieval metrics
    metrics = full_eval(sketch_model, val_loader, brep_anchors, config)

    # Print results
    lr_vit = optimizer.param_groups[0]['lr']
    lr_proj = optimizer.param_groups[1]['lr']
    elapsed = time.time() - t0
    print(f'  Train loss: {train_loss:.4f}  Val loss: {val_loss:.4f}  '
          f'LR: vit={lr_vit:.2e} proj={lr_proj:.2e}  Time: {elapsed:.0f}s')

    r1 = metrics.get(f'sketch2brep_R@1_{config.D_SHARED}d', 0)
    r5 = metrics.get(f'sketch2brep_R@5_{config.D_SHARED}d', 0)
    r10 = metrics.get(f'sketch2brep_R@10_{config.D_SHARED}d', 0)
    print(f'  sketch2brep @ {config.D_SHARED}d: R@1={r1:.2f}% R@5={r5:.2f}% R@10={r10:.2f}%')

    # Track history
    history.append({
        'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss,
        'lr_vit': lr_vit, 'lr_proj': lr_proj, **metrics
    })

    # Save checkpoints
    sb_r1 = metrics.get(f'sketch2brep_R@1_{config.D_SHARED}d', 0)
    is_best = sb_r1 > best_r1

    # Always save latest (lightweight, no optimizer)
    latest_path = os.path.join(config.LOCAL_SAVE_DIR, 'latest.pth')
    torch.save({
        'epoch': epoch, 'model_state_dict': sketch_model.state_dict(),
        'metrics': metrics, 'config': {k: v for k, v in vars(config).items() if not k.startswith('_')},
    }, latest_path)

    if is_best:
        best_r1 = sb_r1
        best_path = os.path.join(config.LOCAL_SAVE_DIR, 'best.pth')
        torch.save({
            'epoch': epoch, 'model_state_dict': sketch_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': metrics,
        }, best_path)
        # Sync to Drive
        if IS_COLAB and config.SAVE_DIR != config.LOCAL_SAVE_DIR:
            import shutil
            os.makedirs(config.SAVE_DIR, exist_ok=True)
            shutil.copy2(best_path, os.path.join(config.SAVE_DIR, 'best.pth'))
        print(f'  *** New best: sketch2brep R@1 = {best_r1:.2f}% ***')

    if (epoch + 1) % config.SAVE_EVERY == 0 and IS_COLAB and config.SAVE_DIR != config.LOCAL_SAVE_DIR:
        import shutil
        os.makedirs(config.SAVE_DIR, exist_ok=True)
        shutil.copy2(latest_path, os.path.join(config.SAVE_DIR, 'latest.pth'))
        print(f'  Synced latest.pth to Drive')

print(f'\nTraining complete. Best sketch2brep R@1: {best_r1:.2f}%')


In [ ]:
# Cell 17: Final Evaluation - Full Matryoshka Matrix

print('=== Final Evaluation ===')
print()

# Load best checkpoint
best_path = os.path.join(config.LOCAL_SAVE_DIR, 'best.pth')
if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    sketch_model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded best checkpoint from epoch {ckpt["epoch"]+1}')
else:
    print('Using current model (no best checkpoint found)')

metrics = full_eval(sketch_model, val_loader, brep_anchors, config)

print(f'\n{"Dim":>6} | {"sketch->brep R@1":>16} {"R@5":>6} {"R@10":>6}')
print('-' * 50)
for dim in config.MATRYOSHKA_DIMS:
    sb1 = metrics.get(f'sketch2brep_R@1_{dim}d', 0)
    sb5 = metrics.get(f'sketch2brep_R@5_{dim}d', 0)
    sb10 = metrics.get(f'sketch2brep_R@10_{dim}d', 0)
    print(f'{dim:>6}d | {sb1:>15.2f}% {sb5:>5.2f}% {sb10:>5.2f}%')

# Print training history summary
print(f'\n=== Training History ===')
for h in history:
    sb = h.get(f'sketch2brep_R@1_{config.D_SHARED}d', 0)
    print(f'  Epoch {h["epoch"]:2d}: train={h["train_loss"]:.4f} val={h["val_loss"]:.4f} '
          f'sketch2brep_R@1={sb:.2f}%')

